# DEX admin

This notebook covers operations that require the caller to be a **controller** of the DEX canister:

1. Upgrading the canister to a new WASM
2. Adding a new trading pair

Calls will fail when run from a non-controller identity.

## Prerequisites

- [`icp` CLI](https://cli.internetcomputer.org/) installed
- Launch this notebook from the **project root** so `--environment staging` resolves the `dex` canister name (defined in `icp.yaml`)
- An identity that is a controller of the DEX canister. The repo convention is the `hsm` identity — adjust `IDENTITY` below if you use a different one.

----

## Setup

### Check your identity is a controller

`canister status` lists the controllers. Your principal must appear there, otherwise every update call in this notebook will be rejected.

In [39]:
IDENTITY = "hsm"   # controller identity; matches the default in the justfile

### Unlock the identity (HSM PIN / PEM password)

Every signing call in this notebook targets `--identity "$IDENTITY"`. If that identity is HSM-linked or password-encrypted, `icp` would otherwise prompt on a TTY that Jupyter can't render.

`--identity-password-file <FILE>` feeds the unlock secret (HSM PIN or PEM password) to `icp` from a file instead of prompting, so every command runs non-interactively.

Two modes are supported below:

- **You already have the PIN in a local file** — set `EXISTING_PIN_FILE` to its path (e.g. `"~/.config/icp/hsm.pin"`). The notebook will just reference it and leave it untouched on cleanup.
- **You don't have such a file** — leave `EXISTING_PIN_FILE = None`. The cell will prompt once via `getpass.getpass()` (no echo, not persisted in the notebook source), write the secret to a chmod-600 temp file, and remove that temp file in the cleanup cell at the bottom.

In [52]:
import os, tempfile, getpass

EXISTING_PIN_FILE = None   # e.g. "~/.config/icp/hsm.pin" to skip the prompt

if EXISTING_PIN_FILE:
    PIN_FILE = os.path.expanduser(EXISTING_PIN_FILE)
    assert os.path.isfile(PIN_FILE), f"not a file: {PIN_FILE}"
    _created_temp = False
    print("using", PIN_FILE)
else:
    _secret = getpass.getpass(f"Unlock {IDENTITY} (HSM PIN or PEM password): ")
    _fd, PIN_FILE = tempfile.mkstemp(prefix="icp-identity-", suffix=".txt")
    with os.fdopen(_fd, "w") as _f:
        _f.write(_secret)
    os.chmod(PIN_FILE, 0o600)
    del _secret
    _created_temp = True
    print("wrote", PIN_FILE)

using /Users/greg/.nitrokey_hsm


In [53]:
!icp identity principal --identity "$IDENTITY" --identity-password-file "$PIN_FILE"
!icp canister status dex --environment staging --identity "$IDENTITY" --identity-password-file "$PIN_FILE"

ezu3d-2mifu-k3bh4-oqhrj-mbrql-5p67r-pp6pr-dbfra-unkx5-sxdtv-rae
Canister Id: proc5-daaaa-aaaar-qb5va-cai
Canister Name: dex
Canister Status Report:
  Status: Running
  Controllers: cpbhu-5iaaa-aaaad-aalta-cai, cmqvo-qqaaa-aaaai-q3waa-cai, mf7xa-laaaa-aaaar-qaaaa-cai, ezu3d-2mifu-k3bh4-oqhrj-mbrql-5p67r-pp6pr-dbfra-unkx5-sxdtv-rae
  Compute allocation: 0
  Memory allocation: 0
  Freezing threshold: 2_592_000
  Reserved cycles limit: 5_000_000_000_000
  Wasm memory limit: 3_221_225_472
  Wasm memory threshold: 0
  Log memory limit: 0
  Log visibility: Controllers
  Environment Variables:
    Name: PUBLIC_CANISTER_ID:dex, Value: proc5-daaaa-aaaar-qb5va-cai
  Module hash: 0xf533c358ef8e9d3f8564ef87504ef72d04932e488d7eb2782c2ef3bae27aec18
  Memory size: 35_310_326
  Cycles: 6_342_596_212_274
  Reserved cycles: 0
  Idle cycles burned per day: 2_359_363_702
  Query stats:
    Calls: 0
    Instructions: 0
    Req payload bytes: 0
    Res payload bytes: 0


---

## Upgrade the canister

### Build the WASM

`just build` compiles `dex_canister` to `wasm32-unknown-unknown` in release mode and produces `wasms/dex_canister.wasm.gz`. The icp CLI picks up that artifact automatically via the recipe declared in `icp.yaml`.

In [7]:
!just build

cargo build --locked --target wasm32-unknown-unknown --release --package dex_canister
   Compiling proc-macro2 v1.0.106
   Compiling unicode-ident v1.0.15
   Compiling quote v1.0.45
   Compiling version_check v0.9.5
   Compiling typenum v1.17.0
   Compiling serde_core v1.0.228
   Compiling serde v1.0.228
   Compiling autocfg v1.4.0
   Compiling thiserror v1.0.69
   Compiling syn v1.0.109
   Compiling rustversion v1.0.19      ] 0/184: serde_core(build.rs), unico…
   Compiling generic-array v0.14.7
   Compiling anyhow v1.0.95           ] 2/184: serde_core(build.rs), rustv…
   Compiling either v1.13.0           ] 5/184: serde_core(build.rs), anyho…
   Compiling cfg-if v1.0.0            ] 8/184: serde_core(build.rs), anyho…
   Compiling libm v0.2.16            ] 13/184: cfg-if, typenum(build), rus…
   Compiling num-traits v0.2.19      ] 16/184: libm(build), typenum(build)…
   Compiling lazy_static v1.5.0      ] 17/184: libm(build), typenum(build)…
   Compiling paste v1.0.15           ] 18/

### Deploy

Two equivalent paths:

**A. Using the `just` recipe** — defers args to the icp CLI defaults (`(null)`), keeps config unchanged:

```bash
just deploy             # uses IDENTITY=hsm
# just deploy <other>   # override the identity
```

**B. Running the full command explicitly** — gives you control over the upgrade arg:

In [ ]:
!icp canister install dex --mode upgrade --args '(null)' --identity "$IDENTITY" --identity-password-file "$PIN_FILE" --environment staging -y

---

## Add a trading pair

`add_trading_pair` is an update call restricted to controllers. The request carries both ledger IDs plus the token metadata (`symbol`, `decimals`). The DEX validates the submitted metadata against what's already registered (if either token is already part of another listed pair) — mismatches return `InconsistentTokenMetadata`.

### Choose the ledgers

Set the two ledger canister IDs. Base is the asset being bought/sold; quote is the asset prices are denominated in.

In [42]:
BASE_LEDGER  = "la34w-haaaa-aaaar-qb5na-cai"   # ckSOL (devnet)
QUOTE_LEDGER = "apia6-jaaaa-aaaar-qabma-cai"   # ckSepoliaETH

### Fetch ledger metadata

The `symbol` and `decimals` you submit **must** match what each ledger reports via `icrc1_symbol` / `icrc1_decimals`. Mismatches cause either the DEX to reject the call outright (if the token is already registered) or, more insidiously, correctly-matching but wrong metadata that misrepresents the asset.

In [43]:
!icp canister call "$BASE_LEDGER"  icrc1_symbol   '()' --network ic --query --identity anonymous
!icp canister call "$BASE_LEDGER"  icrc1_decimals '()' --network ic --query --identity anonymous
!icp canister call "$QUOTE_LEDGER" icrc1_symbol   '()' --network ic --query --identity anonymous
!icp canister call "$QUOTE_LEDGER" icrc1_decimals '()' --network ic --query --identity anonymous

("ckDevnetSOL")
(9 : nat8)
("ckSepoliaETH")
(18 : nat8)


### Choose tick size and lot size

- `tick_size` — the minimum price increment (in quote-token base units per base-token base unit). All order prices must be a positive multiple.
- `lot_size` — the minimum quantity (in base-token base units). All order quantities must be a positive multiple.

Both are `nat64`, both must be > 0, and both are **fixed for the lifetime of the pair**.

#### Referencing an established exchange

Centralized exchanges have already picked these parameters for most major pairs, balancing price precision against spam-order resistance. Binance's public REST endpoint is a convenient sanity check:

```
GET https://api.binance.com/api/v3/exchangeInfo?symbol=<SYMBOL>
```

The response's `filters` array contains a `PRICE_FILTER` (`tickSize`) and a `LOT_SIZE` (`stepSize`). Those values are human-readable decimal token counts — convert to the DEX's integer base units using the ledger decimals from §2.2:

- `tick_size = tickSize_binance × 10^(quote_decimals − base_decimals)`
- `lot_size  = stepSize_binance × 10^base_decimals`

Fetch Binance's `SOLETH` filters with `curl` + `jq`, then set `TICK_SIZE` / `LOT_SIZE` manually from the output.

In [49]:
BASE_SYMBOL    = "ckSOL"           # copy from icrc1_symbol   output above
BASE_DECIMALS  = 9                 # copy from icrc1_decimals output above

QUOTE_SYMBOL   = "ckSepoliaETH"    # copy from icrc1_symbol   output above
QUOTE_DECIMALS = 18                # copy from icrc1_decimals output above

In [20]:
!curl -sSf "https://api.binance.com/api/v3/exchangeInfo?symbol=SOLETH" | jq '{tickSize: (.symbols[0].filters[] | select(.filterType=="PRICE_FILTER") | .tickSize), stepSize: (.symbols[0].filters[] | select(.filterType=="LOT_SIZE") | .stepSize)}'

{
  "tickSize": "0.00001000",
  "stepSize": "0.00100000"
}


In [50]:
# Binance SOLETH  →  ckSOL / ckSepoliaETH  (BASE_DECIMALS=9, QUOTE_DECIMALS=18)
# tickSize = 0.00001 ETH/SOL  →  tick_size = 0.00001 × 10^(18 − 9) = 10_000
# stepSize = 0.001   SOL      →  lot_size  = 0.001   × 10^9        = 1_000_000
TICK_SIZE = 10_000
LOT_SIZE  = 1_000_000

### Call `add_trading_pair`


In [54]:
from pathlib import Path

ARGS = """(
    record {
        base = record {
            id = record { ledger_id = principal "__BASE_LEDGER__" };
            metadata = record { symbol = "__BASE_SYMBOL__"; decimals = __BASE_DECIMALS__ : nat8 }
        };
        quote = record {
            id = record { ledger_id = principal "__QUOTE_LEDGER__" };
            metadata = record { symbol = "__QUOTE_SYMBOL__"; decimals = __QUOTE_DECIMALS__ : nat8 }
        };
        tick_size = __TICK_SIZE__ : nat64;
        lot_size  = __LOT_SIZE__  : nat64
    }
)"""

ARGS = (ARGS
    .replace("__BASE_LEDGER__",    BASE_LEDGER)
    .replace("__QUOTE_LEDGER__",   QUOTE_LEDGER)
    .replace("__BASE_SYMBOL__",    BASE_SYMBOL)
    .replace("__QUOTE_SYMBOL__",   QUOTE_SYMBOL)
    .replace("__BASE_DECIMALS__",  str(BASE_DECIMALS))
    .replace("__QUOTE_DECIMALS__", str(QUOTE_DECIMALS))
    .replace("__TICK_SIZE__",      str(TICK_SIZE))
    .replace("__LOT_SIZE__",       str(LOT_SIZE)))

print(ARGS)

Path("/tmp/icp-args.did").write_text(ARGS)
!icp canister call dex add_trading_pair --args-file /tmp/icp-args.did --identity "$IDENTITY" --identity-password-file "$PIN_FILE" --environment staging

(
    record {
        base = record {
            id = record { ledger_id = principal "la34w-haaaa-aaaar-qb5na-cai" };
            metadata = record { symbol = "ckSOL"; decimals = 9 : nat8 }
        };
        quote = record {
            id = record { ledger_id = principal "apia6-jaaaa-aaaar-qabma-cai" };
            metadata = record { symbol = "ckSepoliaETH"; decimals = 18 : nat8 }
        };
        tick_size = 10000 : nat64;
        lot_size  = 1000000  : nat64
    }
)
(variant { Err = variant { TradingPairAlreadyExists } })


### Verify the listing

The new pair should appear in the `get_trading_pairs` query:

In [47]:
!icp canister call dex get_trading_pairs '()' --environment staging --query --identity anonymous

(
  vec {
    record {
      base = record {
        id = record { ledger_id = principal "la34w-haaaa-aaaar-qb5na-cai" };
        metadata = record { decimals = 9 : nat8; symbol = "ckSOL" };
      };
      quote = record {
        id = record { ledger_id = principal "apia6-jaaaa-aaaar-qabma-cai" };
        metadata = record { decimals = 18 : nat8; symbol = "ckSepoliaETH" };
      };
      lot_size = 1_000_000 : nat64;
      tick_size = 10_000 : nat64;
    };
  },
)


## What's next

- Every `add_trading_pair` is recorded in the append-only event log — inspect with `get_events` (see `canister/dex.did`).
- See `examples/getting_started.ipynb` for how traders interact with a listed pair (deposit → order → withdraw).

## Clean up

Remove the temporary PIN / password file if we created one. An `EXISTING_PIN_FILE` you pointed at is left in place.

In [56]:
import os
if _created_temp and os.path.exists(PIN_FILE):
    os.unlink(PIN_FILE)
    print("removed", PIN_FILE)
else:
    print("left", PIN_FILE, "untouched")

left /Users/greg/.nitrokey_hsm untouched
